### Project Config Steup

In [1]:
# ================================
# Project Configuration
# ================================

from pathlib import Path
import os
import random

import numpy as np
import pandas as pd
import torch

# Hugging Face
from datasets import load_dataset
from huggingface_hub import hf_hub_download

# Visualization
import matplotlib.pyplot as plt

# Display Settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

# -------------------------------
# Reproducibility
# -------------------------------

SUBSAMPLE_SEED = 42

def set_seed(seed=SUBSAMPLE_SEED):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

# -------------------------------
# Project Paths
# -------------------------------

PROJECT_ROOT = Path("/kaggle/working")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"

for folder in [RAW_DIR, PROCESSED_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project initialized")
print(f"Working Directory : {PROJECT_ROOT}")
print(f"GPU : {torch.cuda.get_device_name(0)}")

Project initialized
Working Directory : /kaggle/working
GPU : Tesla T4


### Getting Data

In [2]:
# ==========================================
# Download IndicXNLI (Hindi & Telugu)
# ==========================================

from huggingface_hub import hf_hub_download

REPO_ID = "Divyanshu/indicxnli"

FILES = [

    # Hindi
    "forward/train/xnli_hi.json",
    "forward/dev/xnli_hi.json",
    "forward/test/xnli_hi.json",

    # Telugu
    "forward/train/xnli_te.json",
    "forward/dev/xnli_te.json",
    "forward/test/xnli_te.json",

]

for file in FILES:

    path = hf_hub_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        filename=file,
        local_dir=RAW_DIR,
        local_dir_use_symlinks=False
    )

    print(path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


forward/train/xnli_hi.json:   0%|          | 0.00/219M [00:00<?, ?B/s]

/kaggle/working/data/raw/forward/train/xnli_hi.json


forward/dev/xnli_hi.json:   0%|          | 0.00/1.32M [00:00<?, ?B/s]

/kaggle/working/data/raw/forward/dev/xnli_hi.json


...

/kaggle/working/data/raw/forward/train/xnli_te.json


forward/dev/xnli_te.json:   0%|          | 0.00/1.33M [00:00<?, ?B/s]

/kaggle/working/data/raw/forward/dev/xnli_te.json


forward/test/xnli_te.json:   0%|          | 0.00/2.70M [00:00<?, ?B/s]

/kaggle/working/data/raw/forward/test/xnli_te.json


### JSON Loader

In [3]:
# ==========================================
# Generic IndicXNLI Loader (Final)
# ==========================================

import json
import pandas as pd

def load_xnli(language: str, split: str) -> pd.DataFrame:
    """
    Load IndicXNLI dataset.

    Parameters
    ----------
    language : str
        "hi", "te"

    split : str
        "train", "dev", "test"

    Returns
    -------
    DataFrame
    """

    path = RAW_DIR / "forward" / split / f"xnli_{language}.json"

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # train -> {'train':[...]}
    # dev   -> {'validation':[...]}
    # test  -> {'test':[...]}
    records = next(iter(data.values()))

    df = pd.DataFrame(records)

    df["language"] = language
    df["split"] = split

    return df

In [4]:
hi_train = load_xnli("hi", "train")
hi_dev   = load_xnli("hi", "dev")
hi_test  = load_xnli("hi", "test")

te_train = load_xnli("te", "train")
te_dev   = load_xnli("te", "dev")
te_test  = load_xnli("te", "test")

In [5]:
for name, df in {
    "HI Train": hi_train,
    "HI Dev": hi_dev,
    "HI Test": hi_test,
    "TE Train": te_train,
    "TE Dev": te_dev,
    "TE Test": te_test,
}.items():

    print(name, df.shape)

HI Train (392702, 5)
HI Dev (2490, 5)
HI Test (5010, 5)
TE Train (392702, 5)
TE Dev (2490, 5)
TE Test (5010, 5)


In [6]:
hi_train = load_xnli("hi", "train")
hi_dev   = load_xnli("hi", "dev")
hi_test  = load_xnli("hi", "test")

te_train = load_xnli("te", "train")
te_dev   = load_xnli("te", "dev")
te_test  = load_xnli("te", "test")

In [7]:
# ==========================================
# Merge Hindi & Telugu
# ==========================================

train_df = pd.concat(
    [hi_train, te_train],
    ignore_index=True
)

valid_df = pd.concat(
    [hi_dev, te_dev],
    ignore_index=True
)

test_df = pd.concat(
    [hi_test, te_test],
    ignore_index=True
)

print(train_df.shape)
print(valid_df.shape)
print(test_df.shape)

(785404, 5)
(4980, 5)
(10020, 5)


In [8]:
train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_df["sample_id"] = train_df.index
valid_df["sample_id"] = valid_df.index
test_df["sample_id"] = test_df.index

In [9]:
for df in [
    hi_train, hi_dev, hi_test,
    te_train, te_dev, te_test
]:

    df.reset_index(drop=True, inplace=True)

    df["sample_id"] = df.index
    df["dataset"] = "IndicXNLI"

In [10]:
hi_train["stratify_key"] = hi_train["label"]

te_train["stratify_key"] = te_train["label"]

In [11]:
from sklearn.model_selection import StratifiedShuffleSplit

def create_nested_subsets(df, budgets, seed=42):

    budgets = sorted(budgets, reverse=True)

    subsets = {}

    current = df.copy()

    for budget in budgets:

        if budget >= len(current):
            subsets[budget] = current.copy()
            continue

        splitter = StratifiedShuffleSplit(
            n_splits=1,
            train_size=budget,
            random_state=seed
        )

        idx, _ = next(
            splitter.split(
                current,
                current["label"]
            )
        )

        current = (
            current
            .iloc[idx]
            .reset_index(drop=True)
        )

        subsets[budget] = current.copy()

    return dict(sorted(subsets.items()))

In [12]:
# Training sample sizes
TRAIN_BUDGETS = [50, 100, 500, 1000,len(hi_train)]

In [13]:
hi_subsets = create_nested_subsets(
    hi_train,
    TRAIN_BUDGETS,
    seed=42
)

In [14]:
te_subsets = create_nested_subsets(
    te_train,
    TRAIN_BUDGETS,
    seed=42
)

In [15]:
# ==========================================
# Validate Hindi Subsets
# ==========================================

def validate_subsets(subsets, language):

    for budget, subset in subsets.items():

        print("=" * 60)
        print(f"{language} | Budget : {budget:,}")

        print("\nSamples")
        print(len(subset))

        print("\nLabel Distribution")
        print(subset["label"].value_counts().sort_index())

        print("\n")

In [16]:
validate_subsets(hi_subsets, "Hindi")

Hindi | Budget : 50

Samples
50

Label Distribution
label
0    16
1    17
2    17
Name: count, dtype: int64


Hindi | Budget : 100

Samples
100

Label Distribution
label
0    33
1    33
2    34
Name: count, dtype: int64


Hindi | Budget : 500

Samples
500

Label Distribution
label
0    166
1    167
2    167
Name: count, dtype: int64


Hindi | Budget : 1,000

Samples
1000

Label Distribution
label
0    333
1    333
2    334
Name: count, dtype: int64


Hindi | Budget : 392,702

Samples
392702

Label Distribution
label
0    130899
1    130900
2    130903
Name: count, dtype: int64




In [17]:
validate_subsets(te_subsets, "Telugu")

Telugu | Budget : 50

Samples
50

Label Distribution
label
0    16
1    17
2    17
Name: count, dtype: int64


Telugu | Budget : 100

Samples
100

Label Distribution
label
0    33
1    33
2    34
Name: count, dtype: int64


Telugu | Budget : 500

Samples
500

Label Distribution
label
0    166
1    167
2    167
Name: count, dtype: int64


Telugu | Budget : 1,000

Samples
1000

Label Distribution
label
0    333
1    333
2    334
Name: count, dtype: int64


Telugu | Budget : 392,702

Samples
392702

Label Distribution
label
0    130899
1    130900
2    130903
Name: count, dtype: int64




In [18]:
# ==========================================
# Verify Nested Subsets
# ==========================================

def verify_nested(subsets):

    budgets = sorted(subsets.keys())

    print("=" * 60)
    print("Nested Subset Verification")
    print("=" * 60)

    for i in range(len(budgets)-1):

        small = budgets[i]
        large = budgets[i+1]

        small_ids = set(subsets[small]["sample_id"])
        large_ids = set(subsets[large]["sample_id"])

        print(
            f"{small:>7} ⊂ {large:<7}:",
            small_ids.issubset(large_ids)
        )

In [19]:
print("Hindi")
verify_nested(hi_subsets)

print()

print("Telugu")
verify_nested(te_subsets)

Hindi
Nested Subset Verification
     50 ⊂ 100    : True
    100 ⊂ 500    : True
    500 ⊂ 1000   : True
   1000 ⊂ 392702 : True

Telugu
Nested Subset Verification
     50 ⊂ 100    : True
    100 ⊂ 500    : True
    500 ⊂ 1000   : True
   1000 ⊂ 392702 : True


In [20]:
# ==========================================
# Save Processed Datasets
# ==========================================

from pathlib import Path

HI_DIR = PROCESSED_DIR / "hi"
TE_DIR = PROCESSED_DIR / "te"

HI_DIR.mkdir(parents=True, exist_ok=True)
TE_DIR.mkdir(parents=True, exist_ok=True)

In [21]:
for budget, subset in hi_subsets.items():

    subset.to_parquet(
        HI_DIR / f"train_{budget}.parquet",
        index=False
    )

hi_dev.to_parquet(
    HI_DIR / "valid.parquet",
    index=False
)

hi_test.to_parquet(
    HI_DIR / "test.parquet",
    index=False
)

In [22]:
for budget, subset in te_subsets.items():

    subset.to_parquet(
        TE_DIR / f"train_{budget}.parquet",
        index=False
    )

te_dev.to_parquet(
    TE_DIR / "valid.parquet",
    index=False
)

te_test.to_parquet(
    TE_DIR / "test.parquet",
    index=False
)

In [23]:
# ==========================================
# Create Experiment Manifest
# ==========================================

import json

manifest = {
    "dataset": "IndicXNLI",
    "languages": ["hi", "te"],
    "budgets": TRAIN_BUDGETS,
    "seed": SUBSAMPLE_SEED,
    "sampling": "Nested Stratified",
    "stratification": "label",
    "train_size_per_language": len(hi_train),
    "validation_size_per_language": len(hi_dev),
    "test_size_per_language": len(hi_test),
    "total_train_size": len(train_df),
    "total_validation_size": len(valid_df),
    "total_test_size": len(test_df),
}

manifest_path = PROCESSED_DIR / "manifest.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

print(f"Manifest saved to: {manifest_path}")

Manifest saved to: /kaggle/working/data/processed/manifest.json


In [24]:
# ==========================================
# Dataset Summary
# ==========================================

print("=" * 70)
print("IndicXNLI Preprocessing Completed")
print("=" * 70)

print(f"Languages : {manifest['languages']}")
print(f"Budgets   : {manifest['budgets']}")
print(f"Seed      : {manifest['seed']}")

print("\nFiles Saved")

for language in ["hi", "te"]:

    print(f"\n{language.upper()}")

    for budget in TRAIN_BUDGETS:

        print(f"  train_{budget}.parquet")

    print("  valid.parquet")
    print("  test.parquet")

print("\nManifest : manifest.json")

IndicXNLI Preprocessing Completed
Languages : ['hi', 'te']
Budgets   : [50, 100, 500, 1000, 392702]
Seed      : 42

Files Saved

HI
  train_50.parquet
  train_100.parquet
  train_500.parquet
  train_1000.parquet
  train_392702.parquet
  valid.parquet
  test.parquet

TE
  train_50.parquet
  train_100.parquet
  train_500.parquet
  train_1000.parquet
  train_392702.parquet
  valid.parquet
  test.parquet

Manifest : manifest.json


In [25]:
import pandas as pd

df = pd.read_parquet(PROCESSED_DIR / "hi" / "train_1000.parquet")

print(df.shape)
print(df["label"].value_counts())
print(df["language"].unique())

(1000, 8)
label
2    334
0    333
1    333
Name: count, dtype: int64
['hi']


In [26]:
train_50 = pd.read_parquet(PROCESSED_DIR / "hi" / "train_50.parquet")
train_100 = pd.read_parquet(PROCESSED_DIR / "hi" / "train_100.parquet")

assert set(train_50["sample_id"]).issubset(set(train_100["sample_id"]))

In [27]:
import shutil

shutil.make_archive(
    "/kaggle/working/indicxnli_preprocessed",
    "zip",
    PROCESSED_DIR
)

'/kaggle/working/indicxnli_preprocessed.zip'

In [31]:
import os

DATA_DIR = "/kaggle/input/datasets/yashjaiswal04518/indicxnli-preprocessed-v1"

for root, dirs, files in os.walk(DATA_DIR):
    print(root)
    for f in files:
        print("   ", f)

/kaggle/input/datasets/yashjaiswal04518/indicxnli-preprocessed-v1
    manifest.json
/kaggle/input/datasets/yashjaiswal04518/indicxnli-preprocessed-v1/te
    train_392702.parquet
    train_500.parquet
    train_1000.parquet
    test.parquet
    train_50.parquet
    valid.parquet
    train_100.parquet
/kaggle/input/datasets/yashjaiswal04518/indicxnli-preprocessed-v1/hi
    train_392702.parquet
    train_500.parquet
    train_1000.parquet
    test.parquet
    train_50.parquet
    valid.parquet
    train_100.parquet


In [32]:
import pandas as pd

DATA_DIR = "/kaggle/input/datasets/yashjaiswal04518/indicxnli-preprocessed-v1"

hi_train = pd.read_parquet(f"{DATA_DIR}/hi/train_1000.parquet")
te_train = pd.read_parquet(f"{DATA_DIR}/te/train_1000.parquet")

In [33]:
from pathlib import Path
import pandas as pd


def verify_indicxnli_dataset(data_dir):
    """
    Verify the preprocessed IndicXNLI dataset.

    Checks
    ------
    ✓ Required files exist
    ✓ Dataset shapes
    ✓ Missing values
    ✓ Label distribution
    ✓ Language column
    ✓ sample_id uniqueness
    ✓ Nested subsets
    ✓ Manifest exists

    Parameters
    ----------
    data_dir : str or Path
        Root directory of the processed dataset.
    """

    data_dir = Path(data_dir)

    budgets = [50, 100, 500, 1000, 392702]
    languages = ["hi", "te"]

    print("=" * 75)
    print("IndicXNLI Dataset Verification")
    print("=" * 75)

    # ----------------------------------------------------
    # Manifest
    # ----------------------------------------------------
    manifest = data_dir / "manifest.json"

    if manifest.exists():
        print("✓ manifest.json found")
    else:
        print("✗ manifest.json missing")

    print()

    # ----------------------------------------------------
    # Verify each language
    # ----------------------------------------------------
    for lang in languages:

        print(f"[{lang.upper()}]")

        previous_ids = None

        for budget in budgets:

            path = data_dir / lang / f"train_{budget}.parquet"

            if not path.exists():
                print(f"✗ Missing: {path.name}")
                continue

            df = pd.read_parquet(path)

            print(
                f"  ✓ train_{budget:<6}"
                f" rows={len(df):>7}"
                f"  cols={df.shape[1]}"
            )

            # sample_id uniqueness
            if "sample_id" in df.columns:
                unique = df["sample_id"].is_unique
                print(f"      sample_id unique : {unique}")

            # missing values
            missing = int(df.isna().sum().sum())
            print(f"      missing values   : {missing}")

            # language column
            if "language" in df.columns:
                print(
                    f"      language         : "
                    f"{df['language'].unique().tolist()}"
                )

            # label distribution
            if "label" in df.columns:
                print("      labels")
                print(df["label"].value_counts(normalize=True).round(3))

            # nested subset verification
            if previous_ids is not None:

                if "sample_id" in df.columns:

                    current = set(df["sample_id"])

                    nested = previous_ids.issubset(current)

                    print(f"      nested           : {nested}")

            previous_ids = set(df["sample_id"])

            print()

        # ------------------------------
        # Validation
        # ------------------------------
        for split in ["valid", "test"]:

            path = data_dir / lang / f"{split}.parquet"

            if not path.exists():
                print(f"✗ Missing: {split}.parquet")
                continue

            df = pd.read_parquet(path)

            print(
                f"  ✓ {split:<10}"
                f" rows={len(df):>7}"
                f" cols={df.shape[1]}"
            )

        print("\n" + "-" * 75)

    print("\nVerification completed.")

In [34]:
verify_indicxnli_dataset(DATA_DIR)

IndicXNLI Dataset Verification
✓ manifest.json found

[HI]
  ✓ train_50     rows=     50  cols=8
      sample_id unique : True
      missing values   : 0
      language         : ['hi']
      labels
label
1    0.34
2    0.34
0    0.32
Name: proportion, dtype: float64

  ✓ train_100    rows=    100  cols=8
      sample_id unique : True
      missing values   : 0
      language         : ['hi']
      labels
label
2    0.34
0    0.33
1    0.33
Name: proportion, dtype: float64
      nested           : True

  ✓ train_500    rows=    500  cols=8
      sample_id unique : True
      missing values   : 0
      language         : ['hi']
      labels
label
1    0.334
2    0.334
0    0.332
Name: proportion, dtype: float64
      nested           : True

  ✓ train_1000   rows=   1000  cols=8
      sample_id unique : True
      missing values   : 0
      language         : ['hi']
      labels
label
2    0.334
0    0.333
1    0.333
Name: proportion, dtype: float64
      nested           : True

  ✓ t

In [36]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("/kaggle/input/datasets/yashjaiswal04518/indicxnli-preprocessed-v1")

LANGUAGES = ["hi", "te"]
BUDGETS = [50, 100, 500, 1000, 392702]

print("=" * 75)
print("Summary Statistics")
print("=" * 75)

summary = []

for lang in LANGUAGES:

    train = pd.read_parquet(DATA_DIR / lang / f"train_{max(BUDGETS)}.parquet")
    valid = pd.read_parquet(DATA_DIR / lang / "valid.parquet")
    test = pd.read_parquet(DATA_DIR / lang / "test.parquet")

    summary.append({
        "Language": lang.upper(),
        "Train": len(train),
        "Valid": len(valid),
        "Test": len(test),
        "Columns": train.shape[1],
        "Classes": train["label"].nunique(),
        "Missing Values": int(train.isna().sum().sum()),
        "Unique sample_id": train["sample_id"].is_unique,
    })

summary_df = pd.DataFrame(summary)

print(summary_df.to_string(index=False))

print("\nTraining Budgets")
print("-" * 75)

for b in BUDGETS:
    print(f"{b:>8,} samples")

print("\nOverall Statistics")
print("-" * 75)
print(f"Total Languages      : {len(LANGUAGES)}")
print(f"Total Train Samples  : {summary_df['Train'].sum():,}")
print(f"Total Valid Samples  : {summary_df['Valid'].sum():,}")
print(f"Total Test Samples   : {summary_df['Test'].sum():,}")
print(f"Total Classes        : {summary_df['Classes'].iloc[0]}")

Summary Statistics
Language  Train  Valid  Test  Columns  Classes  Missing Values  Unique sample_id
      HI 392702   2490  5010        8        3               0              True
      TE 392702   2490  5010        8        3               0              True

Training Budgets
---------------------------------------------------------------------------
      50 samples
     100 samples
     500 samples
   1,000 samples
 392,702 samples

Overall Statistics
---------------------------------------------------------------------------
Total Languages      : 2
Total Train Samples  : 785,404
Total Valid Samples  : 4,980
Total Test Samples   : 10,020
Total Classes        : 3
